# Generic Legal Document Structure Parser V2 (Config-Driven)

Pipeline phân tích cấu trúc văn bản quy phạm pháp luật (VBQPPL) Việt Nam theo mô hình **Config-Driven**:
1. **`documents_config.json`**: Cấu hình danh mục văn bản, đường dẫn `.docx` và metadata tương ứng.
2. **`parser_profiles.json`**: Cấu hình các bộ nhận diện Regex và thứ bậc cấu trúc riêng cho từng loại văn bản (Hiến pháp, Bộ luật, Nghị định...).
3. **`ParserVanBanPhapLuat`**: Engine phân tích cấu trúc đệ quy, không phụ thuộc hard-code, hỗ trợ đa tầng phân cấp (`Phần` $\rightarrow$ `Chương` $\rightarrow$ `Mục` $\rightarrow$ `Tiểu mục` $\rightarrow$ `Điều` $\rightarrow$ `Khoản` $\rightarrow$ `Điểm`).
4. **Comprehensive Diagnostics**: Tự động kiểm tra chất lượng dữ liệu: phát hiện trùng lặp Điều, số Điều nhảy vọt, node mồ côi, nội dung rỗng và các dòng tiêu đề khả nghi chưa được parse.
5. **Output**: Xuất trực tiếp sang `du_lieu_phap_luat.json` với trường `document_id` chuẩn hóa ở top-level và trong `metadata`.

In [1]:
import os
import re
import json
import unicodedata
import pathlib
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
import docx

print("Đã tải đầy đủ các thư viện cần thiết.")

Đã tải đầy đủ các thư viện cần thiết.


In [2]:
# Cấu hình đường dẫn thư mục và file
BASE_DIR = pathlib.Path('../data')
CONFIG_PATH = BASE_DIR / 'documents_config.json'
PROFILES_PATH = BASE_DIR / 'parser_profiles.json'
OUTPUT_PATH = BASE_DIR / 'du_lieu_phap_luat.json'

def load_json(file_path: pathlib.Path) -> Any:
    """Đọc dữ liệu từ file JSON."""
    if not file_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file cấu hình tại: {file_path.resolve()}")
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

documents_config = load_json(CONFIG_PATH)
parser_profiles = load_json(PROFILES_PATH)

print(f" Đã tải cấu hình cho {len(documents_config)} văn bản từ: {CONFIG_PATH.name}")
print(f" Đã tải {len(parser_profiles)} parser profiles: {list(parser_profiles.keys())}")

 Đã tải cấu hình cho 5 văn bản từ: documents_config.json
 Đã tải 3 parser profiles: ['default', 'nghi_dinh', 'nghi_quyet']


In [3]:
def read_docx(file_path: pathlib.Path) -> str:
    """Đọc toàn bộ nội dung văn bản từ file Word (.docx)."""
    if not file_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file tại: {file_path.resolve()}")
    doc = docx.Document(file_path)
    return "\n".join([para.text for para in doc.paragraphs])

def chuan_hoa_dong(line: str) -> str:
    """
    Chuẩn hóa một dòng văn bản:
    - Loại bỏ BOM (Byte Order Mark)
    - Chuẩn hóa Unicode về chuẩn NFC
    - Chuẩn hóa khoảng trắng dư thừa
    """
    if not line:
        return ""
    line = line.replace("\ufeff", "")
    line = unicodedata.normalize("NFC", line)
    line = re.sub(r"[ \t]+", " ", line)
    return line.strip()

def chuan_hoa_text(text: str) -> List[str]:
    """Chuyển đổi văn bản thô thành danh sách các dòng đã chuẩn hóa sạch."""
    if not text:
        return []
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    for raw_line in text.split("\n"):
        line = chuan_hoa_dong(raw_line)
        if line:
            lines.append(line)
    return lines

print("Đã tải các hàm đọc file Word và chuẩn hóa văn bản.")

Đã tải các hàm đọc file Word và chuẩn hóa văn bản.


In [4]:
@dataclass
class Diem:
    so: str
    noi_dung: str

    def to_dict(self) -> dict:
        return {
            "loai": "diem",
            "so": self.so,
            "noi_dung": self.noi_dung,
        }

@dataclass
class Khoan:
    so: int
    noi_dung: str
    cac_diem: list[Diem] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "khoan",
            "so": self.so,
            "noi_dung": self.noi_dung,
            "cac_diem": [diem.to_dict() for diem in self.cac_diem],
        }

@dataclass
class Dieu:
    so: int
    tieu_de: Optional[str] = None
    noi_dung: Optional[str] = None
    cac_khoan: list[Khoan] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "dieu",
            "so": self.so,
            "tieu_de": self.tieu_de,
            "noi_dung": self.noi_dung if not self.cac_khoan else None,
            "cac_khoan": [khoan.to_dict() for khoan in self.cac_khoan],
        }

@dataclass
class TieuMuc:
    so: str
    ten: str
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "tieu_muc",
            "so": self.so,
            "ten": self.ten,
            "cac_dieu": [dieu.to_dict() for dieu in self.cac_dieu],
        }

@dataclass
class Muc:
    so: str
    ten: str
    cac_tieu_muc: list[TieuMuc] = field(default_factory=list)
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "muc",
            "so": self.so,
            "ten": self.ten,
            "cac_tieu_muc": [item.to_dict() for item in self.cac_tieu_muc],
            "cac_dieu": [dieu.to_dict() for dieu in self.cac_dieu],
        }

@dataclass
class Chuong:
    so: str
    ten: str
    cac_muc: list[Muc] = field(default_factory=list)
    cac_tieu_muc: list[TieuMuc] = field(default_factory=list)
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "chuong",
            "so": self.so,
            "ten": self.ten,
            "cac_muc": [muc.to_dict() for muc in self.cac_muc],
            "cac_tieu_muc": [item.to_dict() for item in self.cac_tieu_muc],
            "cac_dieu": [dieu.to_dict() for dieu in self.cac_dieu],
        }

@dataclass
class Phan:
    so: str
    ten: str
    cac_chuong: list[Chuong] = field(default_factory=list)
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "phan",
            "so": self.so,
            "ten": self.ten,
            "cac_chuong": [chuong.to_dict() for chuong in self.cac_chuong],
            "cac_dieu": [dieu.to_dict() for dieu in self.cac_dieu],
        }

@dataclass
class VanBan:
    ten: str
    loai_van_ban: Optional[str] = None
    so_van_ban: Optional[str] = None
    phan_mo_dau: list[str] = field(default_factory=list)
    cac_phan: list[Phan] = field(default_factory=list)
    cac_chuong: list[Chuong] = field(default_factory=list)
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "van_ban",
            "ten": self.ten,
            "loai_van_ban": self.loai_van_ban,
            "so_van_ban": self.so_van_ban,
            "phan_mo_dau": self.phan_mo_dau,
            "cac_phan": [phan.to_dict() for phan in self.cac_phan],
            "cac_chuong": [chuong.to_dict() for chuong in self.cac_chuong],
            "cac_dieu": [dieu.to_dict() for dieu in self.cac_dieu],
        }

print("Đã tải đầy đủ các Dataclass mô hình hóa cấu trúc VBQPPL.")

Đã tải đầy đủ các Dataclass mô hình hóa cấu trúc VBQPPL.


In [5]:
class ParserVanBanPhapLuat:
    """
    Parser cấu trúc phân cấp VBQPPL điều khiển bằng Profile (Config-Driven Parser).
    Tự động biên dịch các mẫu Regex và thích ứng linh hoạt với từng loại văn bản.
    """
    def __init__(
        self,
        profile: dict,
        ten_van_ban: str,
        loai_van_ban: Optional[str] = None,
        so_van_ban: Optional[str] = None,
    ):
        self.profile = profile
        self.compiled_patterns = {}
        for level, pattern_str in profile.get("patterns", {}).items():
            if pattern_str:
                self.compiled_patterns[level] = re.compile(pattern_str, re.IGNORECASE)

        self.title_on_next_line = set(profile.get("title_on_next_line", []))
        self.hierarchy = profile.get("hierarchy", [])

        self.van_ban = VanBan(
            ten=ten_van_ban,
            loai_van_ban=loai_van_ban,
            so_van_ban=so_van_ban,
        )

        # Context trạng thái hiện tại
        self.phan_hien_tai: Optional[Phan] = None
        self.chuong_hien_tai: Optional[Chuong] = None
        self.muc_hien_tai: Optional[Muc] = None
        self.tieu_muc_hien_tai: Optional[TieuMuc] = None
        self.dieu_hien_tai: Optional[Dieu] = None
        self.khoan_hien_tai: Optional[Khoan] = None
        self.diem_hien_tai: Optional[Diem] = None

    def parse(self, text: str) -> VanBan:
        lines = chuan_hoa_text(text)
        i = 0
        waiting_for_title = True

        while i < len(lines):
            line = lines[i]

            # 0. Xử lý tiêu đề ở dòng tiếp theo cho Chương hoặc Phần
            if waiting_for_title:
                if self.chuong_hien_tai is not None and self.chuong_hien_tai.ten == "":
                    self.chuong_hien_tai.ten = line.strip()
                    waiting_for_title = False
                    i += 1
                    continue
                elif self.phan_hien_tai is not None and self.phan_hien_tai.ten == "":
                    self.phan_hien_tai.ten = line.strip()
                    waiting_for_title = False
                    i += 1
                    continue

            # 1. PHẦN
            pat_phan = self.compiled_patterns.get("phan")
            if pat_phan:
                match = pat_phan.match(line)
                if match:
                    ten = match.group("ten") if "ten" in match.groupdict() else None
                    self._tao_phan(so=match.group("so"), ten=ten)
                    i += 1
                    if "phan" in self.title_on_next_line and (not ten or not ten.strip()):
                        waiting_for_title = True
                    continue

            # 2. CHƯƠNG
            pat_chuong = self.compiled_patterns.get("chuong")
            if pat_chuong:
                match = pat_chuong.match(line)
                if match:
                    ten = match.group("ten") if "ten" in match.groupdict() else None
                    self._tao_chuong(so=match.group("so"), ten=ten)
                    i += 1
                    if "chuong" in self.title_on_next_line and (not ten or not ten.strip()):
                        waiting_for_title = True
                    continue

            # 3. MỤC
            pat_muc = self.compiled_patterns.get("muc")
            if pat_muc:
                match = pat_muc.match(line)
                if match:
                    ten = match.group("ten") if "ten" in match.groupdict() else None
                    self._tao_muc(so=match.group("so"), ten=ten)
                    i += 1
                    continue

            # 4. TIỂU MỤC
            pat_tieu_muc = self.compiled_patterns.get("tieu_muc")
            if pat_tieu_muc:
                match = pat_tieu_muc.match(line)
                if match:
                    ten = match.group("ten") if "ten" in match.groupdict() else None
                    self._tao_tieu_muc(so=match.group("so"), ten=ten)
                    i += 1
                    continue

            # 5. ĐIỀU
            pat_dieu = self.compiled_patterns.get("dieu")
            if pat_dieu:
                match = pat_dieu.match(line)
                if match:
                    tieu_de = match.group("tieu_de") if "tieu_de" in match.groupdict() else None
                    self._tao_dieu(so=int(match.group("so")), tieu_de=tieu_de)
                    i += 1
                    continue

            # 6. KHOẢN
            pat_khoan = self.compiled_patterns.get("khoan")
            if pat_khoan and self.dieu_hien_tai is not None:
                match = pat_khoan.match(line)
                if match:
                    self._tao_khoan(so=int(match.group("so")), noi_dung=match.group("noi_dung"))
                    i += 1
                    continue

            # 7. ĐIỂM
            pat_diem = self.compiled_patterns.get("diem")
            if pat_diem and self.khoan_hien_tai is not None:
                match = pat_diem.match(line)
                if match:
                    self._tao_diem(so=match.group("so"), noi_dung=match.group("noi_dung"))
                    i += 1
                    continue

            # 8. NỘI DUNG (Multi-line title / Nội dung văn bản)
            self._them_noi_dung(line)
            i += 1

        return self.van_ban

    def _tao_phan(self, so: str, ten: Optional[str]):
        phan = Phan(so=so, ten=ten.strip() if ten else "")
        self.van_ban.cac_phan.append(phan)
        self.phan_hien_tai = phan
        self.chuong_hien_tai = None
        self.muc_hien_tai = None
        self.tieu_muc_hien_tai = None
        self.dieu_hien_tai = None
        self.khoan_hien_tai = None
        self.diem_hien_tai = None

    def _tao_chuong(self, so: str, ten: Optional[str]):
        chuong = Chuong(so=so, ten=ten.strip() if ten else "")
        if self.phan_hien_tai is not None:
            self.phan_hien_tai.cac_chuong.append(chuong)
        else:
            self.van_ban.cac_chuong.append(chuong)
        self.chuong_hien_tai = chuong
        self.muc_hien_tai = None
        self.tieu_muc_hien_tai = None
        self.dieu_hien_tai = None
        self.khoan_hien_tai = None
        self.diem_hien_tai = None

    def _tao_muc(self, so: str, ten: Optional[str]):
        muc = Muc(so=so, ten=ten.strip() if ten else "")
        if self.chuong_hien_tai is not None:
            self.chuong_hien_tai.cac_muc.append(muc)
        self.muc_hien_tai = muc
        self.tieu_muc_hien_tai = None
        self.dieu_hien_tai = None
        self.khoan_hien_tai = None
        self.diem_hien_tai = None

    def _tao_tieu_muc(self, so: str, ten: Optional[str]):
        tieu_muc = TieuMuc(so=so, ten=ten.strip() if ten else "")
        if self.muc_hien_tai is not None:
            self.muc_hien_tai.cac_tieu_muc.append(tieu_muc)
        elif self.chuong_hien_tai is not None:
            self.chuong_hien_tai.cac_tieu_muc.append(tieu_muc)
        self.tieu_muc_hien_tai = tieu_muc
        self.dieu_hien_tai = None
        self.khoan_hien_tai = None
        self.diem_hien_tai = None

    def _tao_dieu(self, so: int, tieu_de: Optional[str]):
        dieu = Dieu(so=so, tieu_de=tieu_de.strip() if tieu_de else None)
        if self.tieu_muc_hien_tai is not None:
            self.tieu_muc_hien_tai.cac_dieu.append(dieu)
        elif self.muc_hien_tai is not None:
            self.muc_hien_tai.cac_dieu.append(dieu)
        elif self.chuong_hien_tai is not None:
            self.chuong_hien_tai.cac_dieu.append(dieu)
        elif self.phan_hien_tai is not None:
            self.phan_hien_tai.cac_dieu.append(dieu)
        else:
            self.van_ban.cac_dieu.append(dieu)
        self.dieu_hien_tai = dieu
        self.khoan_hien_tai = None
        self.diem_hien_tai = None

    def _tao_khoan(self, so: int, noi_dung: str):
        if self.dieu_hien_tai is None:
            return
        khoan = Khoan(so=so, noi_dung=noi_dung)
        self.dieu_hien_tai.cac_khoan.append(khoan)
        self.khoan_hien_tai = khoan
        self.diem_hien_tai = None

    def _tao_diem(self, so: str, noi_dung: str):
        if self.khoan_hien_tai is None:
            return
        diem = Diem(so=so, noi_dung=noi_dung)
        self.khoan_hien_tai.cac_diem.append(diem)
        self.diem_hien_tai = diem

    def _them_noi_dung(self, line: str):
        if self.dieu_hien_tai is None:
            if self.tieu_muc_hien_tai is not None:
                self.tieu_muc_hien_tai.ten += " " + line
            elif self.muc_hien_tai is not None:
                self.muc_hien_tai.ten += " " + line
            elif self.chuong_hien_tai is not None:
                self.chuong_hien_tai.ten += " " + line
            elif self.phan_hien_tai is not None:
                self.phan_hien_tai.ten += " " + line
            else:
                self.van_ban.phan_mo_dau.append(line)
            return

        if self.diem_hien_tai is not None:
            self.diem_hien_tai.noi_dung += " " + line
            return

        if self.khoan_hien_tai is not None:
            self.khoan_hien_tai.noi_dung += " " + line
            return

        if self.dieu_hien_tai.noi_dung:
            self.dieu_hien_tai.noi_dung += " " + line
        else:
            self.dieu_hien_tai.noi_dung = line

print("Đã tải thành công ParserVanBanPhapLuat (Config-Driven Engine).")

Đã tải thành công ParserVanBanPhapLuat (Config-Driven Engine).


In [6]:
def count_nodes_and_validate(item: dict, stats: dict, article_numbers: list):
    """Duyệt đệ quy kiểm tra toàn diện chất lượng cấu trúc parse."""
    loai = item.get("loai")
    if loai:
        stats[loai] = stats.get(loai, 0) + 1
        
    if loai == "dieu":
        so = item.get("so")
        if so is not None:
            article_numbers.append(so)
        noi_dung = item.get("noi_dung")
        cac_khoan = item.get("cac_khoan", [])
        if not noi_dung and not cac_khoan:
            stats["dieu_khong_co_noi_dung"] = stats.get("dieu_khong_co_noi_dung", 0) + 1
            
    elif loai == "khoan":
        noi_dung = item.get("noi_dung")
        if not noi_dung or not noi_dung.strip():
            stats["khoan_khong_co_noi_dung"] = stats.get("khoan_khong_co_noi_dung", 0) + 1
            
    elif loai == "diem":
        noi_dung = item.get("noi_dung")
        if not noi_dung or not noi_dung.strip():
            stats["diem_khong_co_noi_dung"] = stats.get("diem_khong_co_noi_dung", 0) + 1
            
    elif loai in ["chuong", "muc", "tieu_muc", "phan"]:
        ten = item.get("ten")
        if not ten or not ten.strip():
            stats[f"{loai}_ten_rong"] = stats.get(f"{loai}_ten_rong", 0) + 1

    for k, v in item.items():
        if k.startswith("cac_") and isinstance(v, list):
            for child in v:
                if isinstance(child, dict):
                    count_nodes_and_validate(child, stats, article_numbers)

def check_unparsed_candidate_headings(raw_lines: list[str], profile: dict) -> list[str]:
    """Phát hiện các dòng tiêu đề tiềm năng nhưng không khớp Regex của profile."""
    candidate_rx = re.compile(
        r"^(Điều|Khoản|Điểm|Chương|Mục|Tiểu mục|Phần)\s+(\d+|[IVXLCDM]+|[a-zđ]\)|thứ)",
        re.IGNORECASE
    )
    compiled = {lvl: re.compile(p, re.IGNORECASE) for lvl, p in profile.get("patterns", {}).items() if p}
    unparsed = []
    for line in raw_lines:
        if candidate_rx.match(line):
            if not any(rx.match(line) for rx in compiled.values()):
                unparsed.append(line)
    return unparsed

def run_diagnostics(results: list[dict], profiles: dict, raw_texts: dict) -> dict:
    """Hệ thống báo cáo chất lượng và kiểm định lỗi parse toàn diện."""
    report = {}
    for doc in results:
        doc_id = doc.get("document_id")
        meta = doc.get("metadata", {})
        doc_name = meta.get("ten_van_ban", doc_id)
        cau_truc = doc.get("cau_truc", {})
        
        stats = {
            "phan": 0, "chuong": 0, "muc": 0, "tieu_muc": 0,
            "dieu": 0, "khoan": 0, "diem": 0,
            "dieu_khong_co_noi_dung": 0,
            "khoan_khong_co_noi_dung": 0,
            "diem_khong_co_noi_dung": 0
        }
        article_numbers = []
        count_nodes_and_validate(cau_truc, stats, article_numbers)
        
        warnings = []
        if stats["dieu"] == 0:
            warnings.append("⚠ Không có Điều nào được parse!")
        if stats["dieu_khong_co_noi_dung"] > 0:
            warnings.append(f"⚠ {stats['dieu_khong_co_noi_dung']} Điều không có nội dung")
            
        # Kiểm tra trùng lặp số Điều
        seen_art = set()
        dups = []
        for n in article_numbers:
            if n in seen_art:
                dups.append(n)
            seen_art.add(n)
        if dups:
            warnings.append(f"⚠ Trùng số Điều: {dups}")
            
        # Kiểm tra nhảy số Điều bất thường
        if article_numbers:
            expected = list(range(1, len(article_numbers) + 1))
            if article_numbers != expected:
                diffs = []
                for idx, (act, exp) in enumerate(zip(article_numbers, expected)):
                    if act != exp:
                        diffs.append(f"Vị trí {idx+1}: có Điều {act} thay vì {exp}")
                        if len(diffs) >= 3:
                            break
                warnings.append(f"⚠ Số Điều nhảy/bất thường: {', '.join(diffs)} (Tổng số: {len(article_numbers)}, Max: {max(article_numbers)})")

        # Kiểm tra tiêu đề bị bỏ sót
        profile_name = doc.get("parser_profile", "default")
        prof = profiles.get(profile_name, {})
        lines = chuan_hoa_text(raw_texts.get(doc_id, ""))
        unparsed = check_unparsed_candidate_headings(lines, prof)
        if unparsed:
            warnings.append(f"⚠ Tiêu đề khả nghi chưa khớp pattern ({len(unparsed)} dòng): {unparsed[:3]}")

        report[doc_id] = {
            "doc_name": doc_name,
            "stats": stats,
            "warnings": warnings,
            "total_articles": len(article_numbers)
        }
    return report

print("Đã tải đầy đủ hệ thống Diagnostics & Validation sâu.")

Đã tải đầy đủ hệ thống Diagnostics & Validation sâu.


In [7]:
def main():
    print("=" * 70)
    print(" BẮT ĐẦU PIPELINE PARSE CẤU TRÚC VĂN BẢN (CONFIG-DRIVEN V2)")
    print("=" * 70)
    
    results = []
    raw_texts = {}
    
    for cfg in documents_config:
        doc_id = cfg["document_id"]
        file_path = BASE_DIR / cfg["file_path"]
        profile_name = cfg.get("parser_profile", "default")
        profile = parser_profiles.get(profile_name)
        
        if not profile:
            raise ValueError(f"Không tìm thấy profile '{profile_name}' cho văn bản {doc_id}")
            
        meta = dict(cfg.get("metadata", {}))
        meta["document_id"] = doc_id
        
        print(f"\n[1/4] Đọc file Word: {file_path.name}")
        raw_text = read_docx(file_path)
        raw_texts[doc_id] = raw_text
        
        print(f"[2/4] Phân tích cấu trúc theo profile '{profile_name}'...")
        parser = ParserVanBanPhapLuat(
            profile=profile,
            ten_van_ban=meta.get("ten_van_ban", doc_id),
            loai_van_ban=meta.get("loai_van_ban"),
            so_van_ban=meta.get("so_hieu"),
        )
        van_ban = parser.parse(raw_text)
        
        results.append({
            "document_id": doc_id,
            "parser_profile": profile_name,
            "metadata": meta,
            "cau_truc": van_ban.to_dict()
        })
        print(f"       -> Hoàn thành parse: {meta.get('ten_van_ban')}")

    print("\n" + "=" * 70)
    print(" BÁO CÁO KIỂM ĐỊNH CHẤT LƯỢNG (DIAGNOSTICS REPORT)")
    print("=" * 70)
    
    report = run_diagnostics(results, parser_profiles, raw_texts)
    for doc_id, rep in report.items():
        print(f"\n📌 [{doc_id}] {rep['doc_name']}")
        s = rep['stats']
        print(f"   • Phân cấp : Phần={s['phan']}, Chương={s['chuong']}, Mục={s['muc']}, Tiểu mục={s['tieu_muc']}")
        print(f"   • Quy phạm : Điều={s['dieu']}, Khoản={s['khoan']}, Điểm={s['diem']}")
        if rep["warnings"]:
            for w in rep["warnings"]:
                print(f"   • {w}")
        else:
            print("   • Trạng thái :  100% HỢP LỆ (Không có cảnh báo cấu trúc)")

    print("\n" + "=" * 70)
    print(f" XUẤT DỮ LIỆU RA FILE JSON: {OUTPUT_PATH.resolve()}")
    print("=" * 70)
    
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
        
    print(f" Đã lưu thành công {len(results)} văn bản vào {OUTPUT_PATH.name}")

if __name__ == "__main__":
    main()

 BẮT ĐẦU PIPELINE PARSE CẤU TRÚC VĂN BẢN (CONFIG-DRIVEN V2)

[1/4] Đọc file Word: Hien_phap_nam_2013.docx
[2/4] Phân tích cấu trúc theo profile 'default'...
       -> Hoàn thành parse: Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013

[1/4] Đọc file Word: bo_luat_dan_su_91-2015-QH13.docx
[2/4] Phân tích cấu trúc theo profile 'default'...
       -> Hoàn thành parse: Bộ luật Dân sự 2015

[1/4] Đọc file Word: 21_2021_ND-CP_468069.docx
[2/4] Phân tích cấu trúc theo profile 'nghi_dinh'...
       -> Hoàn thành parse: Nghị định 21/2021/NĐ-CP quy định thi hành Bộ luật Dân sự về bảo đảm thực hiện nghĩa vụ

[1/4] Đọc file Word: 19_2019_ND-CP_386324.docx
[2/4] Phân tích cấu trúc theo profile 'nghi_dinh'...
       -> Hoàn thành parse: Nghị định 19/2019/NĐ-CP về họ, hụi, biêu, phường

[1/4] Đọc file Word: 02_2022_NQ-HDTP_483799.docx
[2/4] Phân tích cấu trúc theo profile 'nghi_quyet'...
       -> Hoàn thành parse: Nghị quyết 02/2022/NQ-HĐTP hướng dẫn áp dụng một số quy định của Bộ luật Dân